# Train v4 resistor box detector

Use a GPU runtime and run the cell below. It downloads the public `v4` `resistor_sam3_merged.zip` COCO dataset, trains a one-class torchvision SSDLite320 + MobileNetV3-Large detector, saves resumable checkpoints to Google Drive, evaluates COCO bbox mAP, exports ONNX, and packages the run outputs.

Default training does **not** download pretrained ImageNet/COCO weights and does not use Ultralytics.
After training, run the separate **image test cell** to upload your own photos and inspect detections without retraining.


In [ ]:
import pathlib, runpy, shutil, subprocess, sys
repo = pathlib.Path('/content/resistor_model_detector_train')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Persie0/resistor_model.git', str(repo)], check=True)
sys.path.insert(0, str(repo / 'src'))
print('Repository commit:', subprocess.check_output(['git', '-C', str(repo), 'rev-parse', '--short', 'HEAD'], text=True).strip())
runpy.run_path(str(repo / 'colab' / 'train_resistor_detector_colab.py'), run_name='__main__')


## Test the trained detector on your photos

**Run this cell after training** (or separately once `best.pt` exists in Google Drive). Upload **multiple JPG/PNG photos** and display resistor bounding boxes with confidence scores on each image. The test cell is rerunnable without restarting training or redownloading the dataset.

It uses the best validation checkpoint, exports it to ONNX if necessary, and maps detector coordinates back to the source image. To change the 0.35 confidence threshold, edit `SCORE_THRESHOLD` in `colab/test_resistor_detector_colab.py`. Images are not used to train or modify weights.


In [ ]:
# POST-TRAINING DETECTOR IMAGE TEST — rerunnable independently.
from pathlib import Path
import runpy
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
repo = Path('/content/resistor_detector/resistor_model')
if not (repo / 'colab' / 'test_resistor_detector_colab.py').is_file():
    repo.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ['git', 'clone', '--depth', '1', 'https://github.com/Persie0/resistor_model.git', str(repo)],
        check=True,
    )

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     '-e', str(repo) + '[detection,export]',
     'onnxruntime', 'opencv-python-headless', 'matplotlib'],
    check=True,
)
sys.path.insert(0, str(repo / 'src'))
run_dir = Path('/content/drive/MyDrive/resistor_model/detector-v4-ssdlite320')
best = run_dir / 'best.pt'
onnx = run_dir / 'resistor_detector_ssdlite320.onnx'
if not best.is_file():
    raise FileNotFoundError(
        f'No best checkpoint found: {best}. Finish at least one training epoch first.'
    )
if not onnx.is_file() or best.stat().st_mtime_ns > onnx.stat().st_mtime_ns:
    print('Exporting current best validation checkpoint to ONNX...')
    subprocess.run(
        [sys.executable, '-m', 'resistor_model.export_detection',
         '--checkpoint', str(best), '--output', str(onnx)],
        cwd=str(repo), check=True,
    )
print(f'Testing best model: {onnx}')
runpy.run_path(str(repo / 'colab' / 'test_resistor_detector_colab.py'), run_name='__main__')
